# Table recall with NV-Ingest and LlamaIndex

Flush vector database otherwise every document you've uploaded previously will be there and could interfere with the results

In [ ]:
from pymilvus import MilvusClient

milvus_client = MilvusClient("http://localhost:19530")
milvus_client.drop_collection('nv_ingest_collection')

Run NV-Ingest on bo767 and get and store embeddings

In [ ]:
! nv-ingest-cli \
  --doc=/datasets/bo767/*.pdf \
  --output_directory=../processed_docs/bo767/ \
  --task='extract:{"document_type": "pdf", "extract_method": "pdfium", "extract_text": "false", "extract_tables": "true", "extract_charts": "false", "extract_images": "false"}' \
  --task='embed:{"text": "false", "tables": "true"}' \
  --task='vdb_upload' \
  --client_host=localhost \
  --client_port=7670

In [ ]:
import os
import glob
import logging
import time
from tqdm import tqdm
from collections import defaultdict
import numpy as np
import pandas as pd

from llama_index.core import VectorStoreIndex
from llama_index.vector_stores.milvus import MilvusVectorStore
from llama_index.embeddings.nvidia import NVIDIAEmbedding

# TODO: Add your NVIDIA API key here
os.environ["NVIDIA_API_KEY"] = "<YOUR_NVIDIA_API_KEY>"


Collect results into a dataframe

In [ ]:
lst = sorted(glob.glob('../processed_docs/bo767/structured/*.pdf.metadata.json'))
len(lst)

In [ ]:
dfs = []
for f in tqdm(lst):
    df = pd.read_json(f)
    df['pdf'] = int(os.path.basename(f).split('.')[0])
    dfs.append(df)
dfs = pd.concat(dfs)[['pdf','metadata']]
dfs.shape

In [ ]:
dfs['type'] = dfs.metadata.apply(lambda x:x['content_metadata']['subtype'])
dfs['page'] = dfs.metadata.apply(lambda x:x['content_metadata']['page_number'])
dfs['table_metadata'] = dfs.metadata.apply(lambda x:x['table_metadata'])
dfs['table_content'] = dfs.table_metadata.apply(lambda x:x['table_content'])

In [ ]:
dfs.type.value_counts() 

In [ ]:
df = dfs[dfs.type=='table'].reset_index(drop=True) # leave only tabels for this notebook
df = df[['pdf','page','table_content']]
df['pdf_page'] = df.apply(lambda x:f"{x.pdf}_{x.page}", axis=1)

In [ ]:
df.tail() 

Get test queries with expected result pdf and page

In [ ]:
df_query = pd.read_csv('table_queries_cleaned_235.csv')[['query','pdf','page','table']]
df_query['pdf_page'] = df_query.apply(lambda x: f"{x.pdf}_{x.page}", axis=1)
df_query

Connect LlamaIndex to our milvus microservice and create a retriever

In [ ]:
embed_model = NVIDIAEmbedding(model="NV-Embed-QA")

vector_store = MilvusVectorStore(
    uri="http://localhost:19530",
    collection_name="nv_ingest_collection",
    doc_id_field="pk",
    embedding_field="vector",
    text_key="text",
    dim=1024,
    overwrite=False
)
index = VectorStoreIndex.from_vector_store(vector_store=vector_store, embed_model=embed_model)
retriever = index.as_retriever(similarity_top_k=10)

Use the retriever to calculate recall scores

In [ ]:
hits = defaultdict(list)

for i in tqdm(range(len(df_query))):
    query = df_query['query'][i]
    expected_texts = df[df['pdf_page']==df_query["pdf_page"][i]]["table_content"].to_list()
    retrieved_texts = [node.text for node in retriever.retrieve(query)]

    for k in [1, 3, 5, 10]:
        hits[k].append(any([expected_text in retrieved_texts[:k] for expected_text in expected_texts]))

for k in hits:
    print(f'  - Recall @{k}: {np.mean(hits[k]) :.3f}')
